In [1]:
!pip install requests beautifulsoup4 pandas lxml pdfplumber

In [2]:
import requests
import pandas as pd
import pdfplumber

from bs4 import BeautifulSoup
from io import BytesIO
from urllib.parse import urljoin

In [3]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [4]:
import requests
import urllib3
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://www.kabwecouncil.gov.zm/"

response = requests.get(
    url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print("Website title:", soup.title.get_text(strip=True))

Status code: 200
Website title: Kabwe Municipal Council – Kabwe


In [5]:
publications_url = "https://www.kabwecouncil.gov.zm/?page_id=195"

response = requests.get(
    publications_url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.get_text(strip=True))

Status code: 200
Publications – Kabwe Municipal Council


In [6]:
budget_links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = link["href"]

    if any(word in text.lower() for word in [
        "budget",
        "financial",
        "finance"
    ]):
        budget_links.append({
            "title": text,
            "url": href
        })

budget_df = pd.DataFrame(budget_links)

budget_df

,title,url
0,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
1,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
2,2018 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
3,2019 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
4,2020 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
5,2021 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
6,2022 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
7,2023 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
8,2024 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,2025 BI Annual Budget Performance Report,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [7]:
financial_statements_df = budget_df[budget_df['title'].str.contains('Financial Statement', case=False, na=False)]
display(financial_statements_df)

,title,url
2,2018 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
3,2019 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
4,2020 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
5,2021 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
6,2022 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
7,2023 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
8,2024 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [8]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

budget_df

,title,url
0,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
1,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
2,2018 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/FS-2018.pdf
3,2019 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/FS-2019.pdf
4,2020 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/FS-2020.pdf
5,2021 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/10/FS-2021.pdf
6,2022 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Financial-Statement-2022.pdf
7,2023 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2023-FINANCIAL-STATEMENTS-KMC-1.pdf
8,2024 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/10/KABWE-M-COUNCIL-2024-APPROVED-FINANCIAL-STATEMENTS.pdf
9,2025 BI Annual Budget Performance Report,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf


In [9]:
budget_row = budget_df[
    budget_df["title"].str.contains(
        "2025 Kabwe Municipal Council OBB Approved Budget",
        case=False,
        na=False
    )
]

budget_row

,title,url
12,2025 Kabwe Municipal Council OBB Approved Budget,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf


In [10]:
budget_url = budget_row["url"].iloc[0]

print("Budget URL:")
print(budget_url)

Budget URL:
https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf


In [11]:
budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("File size:", len(budget_response.content), "bytes")

Status code: 200
File size: 792438 bytes


In [12]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(
    BytesIO(budget_response.content)
)

print("Number of pages:", len(pdf.pages))

Number of pages: 54


In [13]:
for page_number in range(min(5, len(pdf.pages))):
    page = pdf.pages[page_number]

    print(f"\n========== PAGE {page_number + 1} ==========")

    text = page.extract_text()

    if text:
        print(text[:3000])
    else:
        print("No text extracted from this page.")


========== PAGE 1 ==========
OUTPUT BASED ANNUAL BUDGET Page 1
HEA 920 KABWE MUNICIPAL COUNCIL
D 5
1.0 MANDATE
To provide operational and service excellence, innovation, community engagement and observance of
good financial management and accountability. This is in agreement with the Republican Constitution
(Amendment) Act No.2 of 2016 Part IX on the System of Devolved Governance [Article 147 (2)] and Part
XI on the System of Local Government.
2.0 STRATEGY
Kabwe Municipal Council will focus on delivering value and quality of life through good governance and
team work, involving all stakeholders including community representatives through operationalisation
of the Ward Development Committees. Further, in response to the high urbanisation rate, the Local
Authority opened up new areas for development in 2024
3.0 NATIONAL DEVELOPMENT PLAN FRAMEWORK
Cluster : 01 Economic Transformation and Job Creation
Cluster Outcome 01 An Industrialised and Diversified Economy
Strategy : 01 Improve agric

In [14]:
all_tables = []

for page_number, page in enumerate(pdf.pages, start=1):

    tables = page.extract_tables()

    print(f"Page {page_number}: {len(tables)} table(s) found")

    for table in tables:
        if table:
            all_tables.append({
                "page": page_number,
                "table": table
            })

print("\nTotal tables found:", len(all_tables))

Page 1: 7 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 3 table(s) found
Page 6: 0 table(s) found
Page 7: 1 table(s) found
Page 8: 1 table(s) found
Page 9: 1 table(s) found
Page 10: 0 table(s) found
Page 11: 1 table(s) found
Page 12: 2 table(s) found
Page 13: 1 table(s) found
Page 14: 2 table(s) found
Page 15: 2 table(s) found
Page 16: 2 table(s) found
Page 17: 2 table(s) found
Page 18: 3 table(s) found
Page 19: 1 table(s) found
Page 20: 1 table(s) found
Page 21: 2 table(s) found
Page 22: 1 table(s) found
Page 23: 1 table(s) found
Page 24: 1 table(s) found
Page 25: 2 table(s) found
Page 26: 1 table(s) found
Page 27: 2 table(s) found
Page 28: 1 table(s) found
Page 29: 2 table(s) found
Page 30: 1 table(s) found
Page 31: 1 table(s) found
Page 32: 1 table(s) found
Page 33: 1 table(s) found
Page 34: 2 table(s) found
Page 35: 1 table(s) found
Page 36: 2 table(s) found
Page 37: 2 table(s) found
Page 38: 2 table(s) found
Page 39: 1 table(s) f

In [15]:
for item in all_tables[:5]:

    print("\n" + "=" * 80)
    print("PAGE:", item["page"])

    for row in item["table"][:10]:
        print(row)


PAGE: 1
['Improve agricultural production and productivity']
['Promote value addition and manufacturing']
['Improve transport and logistics']
['Enhance the management of petroleum products']

PAGE: 1
['Promote local and diaspora participation in the economy']
['Promote Enterprise development']
['Promote Financial Inclusion']

PAGE: 1
['Enhance access to quality, equitable and inclusive education']
['Increased access to higher education']

PAGE: 1
['Strengthen Public health']
['Increase access to quality health care']

PAGE: 1
['Enhance welfare and livelihoods of poor and vulnerable people']
['Reduce vulnerability associated with HIV and AIDS']


In [16]:
page2 = pdf.pages[1]

table_settings = {
    "vertical_strategy": "text",
    "horizontal_strategy": "text",
    "intersection_tolerance": 5
}

table = page2.extract_table(table_settings)

if table:
    for row in table:
        print(row)
else:
    print("No table detected.")

['Pag', 'e 2', 'OUTPUT BASED ANNUAL B', 'UDGET', '', '']
['', '', '', '', '', '']
['HE', 'A 920', 'KABWE MUNICIPAL COUNCIL', '', '', '']
['D', '5', '', '', '', '']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', 'CODE', 'REVENUE DESCRIPTION', 'APPROVED', 'REVISED', 'BUDGET']
['', '', 'B', 'UDGET 2025\nB', 'UDGET 2026 E', 'STIMATE 2']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', '01', 'Local taxes/rates', '', '', '']
['', '', '', '', '', '']
['', '001', 'Residential', '4,453,818', '4,453,818', '4,453,818']
['', '', '', '', '', '']
['', '002', 'Commercial', '5,536,418', '5,536,418', '5,536,418']
['', '', '', '', '', '']
['', '003', 'Industrial', '2,635,280', '2,635,280', '2,635,280']
['', '', '', '', '', '']
['', '004', 'Hospitality', '572,837', '630,121', '693,133']
['', '', '', '', '', '']
['', '', 'SubItem Total', '13,198,353', '13,255,637', '13,318,649']
['', '001', 'Personal levy', '450,000', '495,000', '220,000']
['', '', '', '', '', '']
['', '', 'SubItem Total', '

In [17]:
# Pages 2 to 5 contain the main revenue and budget tables
budget_rows = []

for page_number in range(2, 6):
    page = pdf.pages[page_number - 1]

    table = page.extract_table({
        "vertical_strategy": "text",
        "horizontal_strategy": "text",
        "intersection_tolerance": 5
    })

    if table:
        for row in table:
            budget_rows.append({
                "page": page_number,
                "row": row
            })

print("Total extracted rows:", len(budget_rows))

Total extracted rows: 269


In [18]:
for item in budget_rows[:40]:
    print(item["page"], item["row"])

2 ['Pag', 'e 2', 'OUTPUT BASED ANNUAL B', 'UDGET', '', '']
2 ['', '', '', '', '', '']
2 ['HE', 'A 920', 'KABWE MUNICIPAL COUNCIL', '', '', '']
2 ['D', '5', '', '', '', '']
2 ['', '', '', '', '', '']
2 ['', '', '', '', '', '']
2 ['', 'CODE', 'REVENUE DESCRIPTION', 'APPROVED', 'REVISED', 'BUDGET']
2 ['', '', 'B', 'UDGET 2025\nB', 'UDGET 2026 E', 'STIMATE 2']
2 ['', '', '', '', '', '']
2 ['', '', '', '', '', '']
2 ['', '01', 'Local taxes/rates', '', '', '']
2 ['', '', '', '', '', '']
2 ['', '001', 'Residential', '4,453,818', '4,453,818', '4,453,818']
2 ['', '', '', '', '', '']
2 ['', '002', 'Commercial', '5,536,418', '5,536,418', '5,536,418']
2 ['', '', '', '', '', '']
2 ['', '003', 'Industrial', '2,635,280', '2,635,280', '2,635,280']
2 ['', '', '', '', '', '']
2 ['', '004', 'Hospitality', '572,837', '630,121', '693,133']
2 ['', '', '', '', '', '']
2 ['', '', 'SubItem Total', '13,198,353', '13,255,637', '13,318,649']
2 ['', '001', 'Personal levy', '450,000', '495,000', '220,000']
2 ['', '

In [19]:
import re
import pandas as pd

records = []
current_category = None

for item in budget_rows:

    page_number = item["page"]
    row = item["row"]

    if not row or len(row) < 6:
        continue

    code = row[1]
    description = row[2]

    # Clean empty cells
    code = code.strip() if code else ""
    description = description.strip() if description else ""

    # Identify main revenue categories such as:
    # 01 Local taxes/rates
    # 02 Fees and Charges
    # 03 Licenses
    # etc.
    if re.fullmatch(r"\d{2}", code) and description:
        current_category = description
        continue

    # Ignore subtotal rows
    if "SubItem Total" in description:
        continue

    # Identify actual revenue items such as 001 Residential
    if re.fullmatch(r"\d{3}", code):

        values = []

        for value in row[3:6]:
            if value:
                value = value.replace(",", "").strip()

                try:
                    values.append(float(value))
                except ValueError:
                    values.append(None)
            else:
                values.append(None)

        # Only keep rows where we have budget figures
        if len(values) == 3 and any(v is not None for v in values):

            records.append({
                "page": page_number,
                "revenue_category": current_category,
                "revenue_code": code,
                "revenue_description": description,
                "budget_2025": values[0],
                "revised_budget_2026": values[1],
                "estimate_2027": values[2]
            })

df_revenue = pd.DataFrame(records)

df_revenue.head(20)

,page,revenue_category,revenue_code,revenue_description,budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
5,4,Licenses,002,Liquor licence,NaN,366000.0,402600.0
6,4,Licenses,003,Firearm and ammuniti,NaN,26000.0,26600.0
7,4,Licenses,004,Petroleum Storage lice,NaN,400000.0,440000.0
8,4,Licenses,005,Dog licence,NaN,40000.0,44000.0
9,4,Levies,001,Livestock Movement le,NaN,32400.0,32400.0


In [20]:
print("Number of revenue records:", len(df_revenue))

df_revenue

Number of revenue records: 34


,page,revenue_category,revenue_code,revenue_description,budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
5,4,Licenses,002,Liquor licence,NaN,366000.0,402600.0
6,4,Licenses,003,Firearm and ammuniti,NaN,26000.0,26600.0
7,4,Licenses,004,Petroleum Storage lice,NaN,400000.0,440000.0
8,4,Licenses,005,Dog licence,NaN,40000.0,44000.0
9,4,Levies,001,Livestock Movement le,NaN,32400.0,32400.0


In [21]:
df_revenue["revenue_category"].unique()

<StringArray>
['Local taxes/rates', 'Licenses', 'Levies', 'Permits', 'Charges']
Length: 5, dtype: str

In [22]:
local_revenue_categories = [
    "Local taxes/rates",
    "Fees and Charges",
    "Licenses",
    "Levies",
    "Permits",
    "Charges",
    "Other Incomes"
]

local_revenue_df = df_revenue[
    df_revenue["revenue_category"].isin(local_revenue_categories)
].copy()

print("Local revenue records:", len(local_revenue_df))

local_revenue_df.head(20)

Local revenue records: 34


,page,revenue_category,revenue_code,revenue_description,budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
5,4,Licenses,002,Liquor licence,NaN,366000.0,402600.0
6,4,Licenses,003,Firearm and ammuniti,NaN,26000.0,26600.0
7,4,Licenses,004,Petroleum Storage lice,NaN,400000.0,440000.0
8,4,Licenses,005,Dog licence,NaN,40000.0,44000.0
9,4,Levies,001,Livestock Movement le,NaN,32400.0,32400.0


In [23]:
local_revenue_file = "db-unza26-csc4792-kabwe-local-revenue-streams.csv"

local_revenue_df.to_csv(
    local_revenue_file,
    sep="|",
    index=False
)

print("Saved:", local_revenue_file)

Saved: db-unza26-csc4792-kabwe-local-revenue-streams.csv


In [24]:
from google.colab import files

files.download(local_revenue_file)

ModuleNotFoundError: No module named 'google'

In [ ]:
import os

print(os.path.abspath(local_revenue_file))
print("File exists:", os.path.exists(local_revenue_file))

In [ ]:
print(local_revenue_df.shape)
display(local_revenue_df.head(10))

In [ ]:
test_df = pd.read_csv(
    local_revenue_file,
    sep="|"
)

display(test_df.head())

In [ ]:
# Check the extracted budget data
display(df_revenue.head())
print("Rows:", len(df_revenue))
print("Columns:", df_revenue.columns.tolist())

In [ ]:
for page_number in range(1, len(pdf.pages) + 1):
    page = pdf.pages[page_number - 1]
    text = page.extract_text() or ""
    
    print(f"\n========== PAGE {page_number} ==========")
    print(text[:2000])

In [ ]:
# Extract the Budget Allocation by Economic Classification section

page5 = pdf.pages[4]

text = page5.extract_text()

print(text[2500:5000])

In [ ]:
# Display all text extracted from page 5
page5 = pdf.pages[4]

text = page5.extract_text()

print(text)

In [ ]:
import pandas as pd

approved_budget_data = [
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Personal Emoluments",
        "budget_amount": 48701693
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Goods and Services",
        "budget_amount": 33797534
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Grants and Other Payments (Transfers)",
        "budget_amount": 34845939
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Non-Financial Assets",
        "budget_amount": 52204514
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Financial Assets",
        "budget_amount": 7473871
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Current Liabilities",
        "budget_amount": 4056000
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Head Total",
        "budget_amount": 181079551
    }
]

df_approved_budget = pd.DataFrame(approved_budget_data)

display(df_approved_budget)

In [ ]:
approved_budget_file = "db-unza26-csc4792-kabwe-approved-budgets.csv"

df_approved_budget.to_csv(
    approved_budget_file,
    sep="|",
    index=False
)

print("Saved:", approved_budget_file)

In [ ]:
import os

print("File exists:", os.path.exists(approved_budget_file))
print("File path:", os.path.abspath(approved_budget_file))

In [ ]:
lgef_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

lgef_response = requests.get(
    lgef_url,
    timeout=60,
    verify=False
)

print("Status code:", lgef_response.status_code)
print("File size:", len(lgef_response.content), "bytes")

In [ ]:
lgef_pdf = pdfplumber.open(BytesIO(lgef_response.content))

print("Number of pages:", len(lgef_pdf.pages))

In [ ]:
# Search the report for LGEF-related information

matches = []

for page_number, page in enumerate(lgef_pdf.pages, start=1):
    text = page.extract_text() or ""
    
    if (
        "LGEF" in text.upper()
        or "LOCAL GOVERNMENT EQUALISATION FUND" in text.upper()
        or "EQUALISATION" in text.upper()
    ):
        matches.append({
            "page": page_number,
            "text": text
        })

print("Pages containing LGEF information:", [m["page"] for m in matches])

In [ ]:
for m in matches:
    print("\n" + "=" * 80)
    print("PAGE:", m["page"])
    print(m["text"][:4000])

In [ ]:
# Check the first 10 pages only
for page_number in range(1, 11):
    text = lgef_pdf.pages[page_number - 1].extract_text() or ""
    
    if "LGEF" in text.upper() or "EQUALISATION" in text.upper():
        print("\n========== PAGE", page_number, "==========")
        print(text[:2000])

In [ ]:
for page_number in range(1, 37):
    print(page_number, end=" ")
    text = lgef_pdf.pages[page_number - 1].extract_text()
    print("OK" if text else "NO TEXT")

In [ ]:
for page_number in range(1, 6):
    page = lgef_pdf.pages[page_number - 1]
    image = page.to_image(resolution=100)
    image.save(f"lgef_page_{page_number}.png")
    print(f"Saved page {page_number}")

In [ ]:
from PIL import Image, ImageOps, ImageDraw

images = []

for page_number in range(1, 6):
    img = Image.open(f"lgef_page_{page_number}.png").convert("RGB")
    img.thumbnail((500, 700))
    images.append(img)

canvas = Image.new("RGB", (1000, 1400), "white")

positions = [
    (0, 0), (500, 0),
    (0, 700), (500, 700),
    (250, 350)
]

for img, pos in zip(images, positions):
    canvas.paste(img, pos)

canvas.save("lgef_pages_1_to_5.png")

print("Saved: lgef_pages_1_to_5.png")

In [ ]:
from IPython.display import display
from PIL import Image

img = Image.open("lgef_pages_1_to_5.png")
display(img)

In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open("lgef_page_1_large.png"))

In [ ]:
from PIL import Image
from IPython.display import display

# Create the large image for page 1
page = lgef_pdf.pages[0]
image = page.to_image(resolution=150)
image.save("lgef_page_1_large.png")

# Display it
display(Image.open("lgef_page_1_large.png"))

In [ ]:
from PIL import Image
from IPython.display import display

# Display the first 5 pages
for i in range(5):
    page = lgef_pdf.pages[i]
    image = page.to_image(resolution=150)

    # Save each page image
    image.save(f"lgef_page_{i+1}_large.png")

    # Display the page
    print(f"--- Page {i+1} ---")
    display(image)

In [ ]:
# Extract text from all pages
all_text = ""

for i, page in enumerate(lgef_pdf.pages):
    text = page.extract_text()
    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print(all_text[:10000])

In [ ]:
import pytesseract

print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
%pip install pytesseract

In [ ]:
import pytesseract

print("pytesseract imported successfully")
print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
import pytesseract

# Tell Python where Tesseract is installed
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
from PIL import Image
import pytesseract

# Load the saved Page 1 image
page_image = Image.open("lgef_page_1_large.png")

# Extract text using OCR
page1_text = pytesseract.image_to_string(page_image)

print(page1_text)

In [ ]:
all_text = ""

for i in range(36):
    image = Image.open(f"lgef_page_{i+1}_large.png")
    text = pytesseract.image_to_string(image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

with open("lgef_financial_statements_ocr.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print("OCR extraction completed and saved.")

In [ ]:
from PIL import Image
from IPython.display import display

# Create and save images for all 36 pages
for i, page in enumerate(lgef_pdf.pages):
    image = page.to_image(resolution=150)
    image.save(f"lgef_page_{i+1}_large.png")

print("All 36 page images saved successfully.")

In [ ]:
import lighteval as lgef

In [ ]:
lgef_pdf = ...

In [ ]:
import pytesseract
from PIL import Image

# Make sure Python knows where Tesseract is installed
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

all_text = ""

for i in range(36):
    image = Image.open(f"lgef_page_{i+1}_large.png")
    text = pytesseract.image_to_string(image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print("OCR extraction completed.")
print(all_text[:10000])

In [ ]:
from PIL import Image
import pytesseract

# Tell Python where Tesseract is installed
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

all_text = ""

# Process all 36 pages directly from the PDF
for i, page in enumerate(lgef_pdf.pages):
    print(f"Processing page {i+1} of {len(lgef_pdf.pages)}...")

    # Render the page as an image
    image = page.to_image(resolution=150)

    # Convert to a PIL Image if necessary
    pil_image = image.original if hasattr(image, "original") else image

    # Run OCR
    text = pytesseract.image_to_string(pil_image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print("OCR extraction completed!")
print(all_text[:10000])

In [ ]:
import pdfplumber

# Open the Kabwe Municipal Council financial statements PDF
pdf_path = "lgef_financial_statements_2025.pdf"

lgef_pdf = pdfplumber.open(pdf_path)

print("PDF opened successfully!")
print("Number of pages:", len(lgef_pdf.pages))

In [ ]:
import requests

url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

response = requests.get(url)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

with open("lgef_financial_statements_2025.pdf", "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully!")

In [ ]:
import tkinter as tk
from tkinter import filedialog
import pdfplumber

# Open a file picker
root = tk.Tk()
root.withdraw()

pdf_path = filedialog.askopenfilename(
    title="Select the Kabwe Municipal Council PDF",
    filetypes=[("PDF files", "*.pdf")]
)

root.destroy()

print("Selected file:", pdf_path)

# Open the selected PDF
lgef_pdf = pdfplumber.open(pdf_path)

print("PDF opened successfully!")
print("Number of pages:", len(lgef_pdf.pages))

In [ ]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

page = lgef_pdf.pages[0]

# Render Page 1
image = page.to_image(resolution=150).original

# Extract text
page1_text = pytesseract.image_to_string(image)

print(page1_text)

In [ ]:
all_text = ""

for i, page in enumerate(lgef_pdf.pages):
    print(f"Processing page {i+1} of {len(lgef_pdf.pages)}...")

    image = page.to_image(resolution=150).original
    text = pytesseract.image_to_string(image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print("OCR extraction completed!")

In [ ]:
import requests
import pdfplumber
import pytesseract
from pathlib import Path

# Official Kabwe Municipal Council PDF
url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

# Save the PDF in the current Jupyter folder
pdf_path = Path.cwd() / "kabwe_financial_statements_2025.pdf"

# Download the PDF if it is not already saved
if not pdf_path.exists():
    print("Downloading PDF...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
    print("PDF downloaded successfully.")

# Open the PDF
lgef_pdf = pdfplumber.open(str(pdf_path))

print("PDF opened successfully!")
print("PDF location:", pdf_path)
print("Number of pages:", len(lgef_pdf.pages))

# Set the Tesseract OCR path
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

# Convert the first page into an image
page = lgef_pdf.pages[0]
page_image = page.to_image(resolution=150)
pil_image = page_image.original

# Extract text using OCR
page1_text = pytesseract.image_to_string(pil_image)

print("\n========== PAGE 1 OCR TEXT ==========\n")
print(page1_text)

In [ ]:
print("Jupyter is working")


In [ ]:
print("Starting PDF setup...")

In [ ]:
import requests

url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

response = requests.get(url, timeout=60)

print("Status code:", response.status_code)
print("Downloaded bytes:", len(response.content))

In [ ]:
pdfplumber.open("lgef_financial_statements_2025.pdf")

In [ ]:
import requests
import pdfplumber
from io import BytesIO

financial_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

financial_response = requests.get(
    financial_url,
    timeout=60,
    verify=False
)

print("Status code:", financial_response.status_code)
print("File size:", len(financial_response.content), "bytes")

financial_pdf = pdfplumber.open(
    BytesIO(financial_response.content)
)

print("Number of pages:", len(financial_pdf.pages))

In [ ]:
financial_text = ""

for page_number, page in enumerate(financial_pdf.pages, start=1):
    page_text = page.extract_text() or ""
    financial_text += f"\n\n--- Page {page_number} ---\n{page_text}"

print(financial_text[:5000])

In [ ]:
from pathlib import Path

pdf_path = Path("financial_statements_2025.pdf")

pdf_path.write_bytes(financial_response.content)

print("Saved to:", pdf_path.resolve())

financial_pdf = pdfplumber.open(pdf_path)

print("Number of pages:", len(financial_pdf.pages))

In [ ]:
pdfplumber.open("financial_statements_2025.pdf")

In [ ]:
print("Number of pages:", len(financial_pdf.pages))

In [ ]:
financial_text = []

for page_number, page in enumerate(financial_pdf.pages, start=1):
    text = page.extract_text() or ""

    financial_text.append({
        "page": page_number,
        "text": text
    })

print("Pages extracted:", len(financial_text))
print(financial_text[0]["text"][:3000])

In [ ]:
financial_text_df = pd.DataFrame(financial_text)

financial_text_df.head()

In [ ]:
page = fininacial_pdf.pages[0]
text=page.extract_text() or""
print(text[:3000])

In [ ]:
page = financial_pdf.pages[0]
text = page.extract_text() or ""

print(text[:3000])

In [ ]:
import requests
import pdfplumber
import pandas as pd
from io import BytesIO

# PDF URL
financial_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

# Download the PDF
response = requests.get(
    financial_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

# Open the downloaded PDF
financial_pdf = pdfplumber.open(
    BytesIO(response.content)
)

print("Number of pages:", len(financial_pdf.pages))

# Extract text from the first page
first_page = financial_pdf.pages[0]
first_page_text = first_page.extract_text() or ""

print(first_page_text[:3000])

In [1]:
!pip install requests pandas pdfplumber

In [2]:
import requests
import pandas as pd
import pdfplumber
from io import BytesIO

In [3]:
pdf_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

download_result = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", download_result.status_code)
print("File size:", len(download_result.content), "bytes")

C:\Users\Mutofwe\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status code: 200
File size: 11368536 bytes


In [4]:
import urllib3

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

In [5]:
pdf_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

download_result = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", download_result.status_code)
print("File size:", len(download_result.content), "bytes")

ChunkedEncodingError: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [6]:
import os

print(os.getcwd())

C:\Users\Mutofwe\group6_administration\notebooks


In [15]:
import pdfplumber

opened_document = pdfplumber.open(
    r"C:\Users\Mutofwe\group6_administration\notebooks\financial_statements_2025.pdf.pdf"
)

print("Number of pages:", len(opened_document.pages))

Number of pages: 36


In [16]:
all_pages_text = ""

for page_number, page in enumerate(opened_document.pages, start=1):
    page_text = page.extract_text()

    if page_text:
        all_pages_text += f"\n\n===== PAGE {page_number} =====\n\n"
        all_pages_text += page_text

print("Total characters extracted:", len(all_pages_text))

Total characters extracted: 0


In [17]:
!pip install pytesseract pillow pdf2image

In [18]:
import pytesseract
from PIL import Image
from pdf2image import convert_from_path

In [19]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

In [20]:
pdf_file_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_statements_2025.pdf.pdf"
)

pdf_images = convert_from_path(
    pdf_file_path,
    dpi=200
)

print("Number of converted pages:", len(pdf_images))

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [1]:
from pdf2image import convert_from_path

pdf_file_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_statements_2025.pdf.pdf"
)

pdf_images = convert_from_path(
    pdf_file_path,
    dpi=200
)

print("Number of converted pages:", len(pdf_images))

Number of converted pages: 36


In [4]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print(pytesseract.get_tesseract_version())

5.5.3.20260724


In [5]:
first_page_ocr = pytesseract.image_to_string(pdf_images[0])

print(first_page_ocr)

KABWE MUNICIPAL COUNCIL.
BI-ANNUAL FINANCLAL STATEMENTS FOR THE YEAR 2025

P O Box 80424
Civic Centre

KABWE




In [6]:
all_pages_text = ""

for page_number, page_image in enumerate(pdf_images, start=1):
    print(f"Processing page {page_number} of {len(pdf_images)}...")

    page_text = pytesseract.image_to_string(page_image)

    all_pages_text += (
        f"\n\n===== PAGE {page_number} =====\n\n"
        + page_text
    )

print("Total characters extracted:", len(all_pages_text))

Processing page 1 of 36...
Processing page 2 of 36...
Processing page 3 of 36...
Processing page 4 of 36...
Processing page 5 of 36...
Processing page 6 of 36...
Processing page 7 of 36...
Processing page 8 of 36...
Processing page 9 of 36...
Processing page 10 of 36...
Processing page 11 of 36...
Processing page 12 of 36...
Processing page 13 of 36...
Processing page 14 of 36...
Processing page 15 of 36...
Processing page 16 of 36...
Processing page 17 of 36...
Processing page 18 of 36...
Processing page 19 of 36...
Processing page 20 of 36...
Processing page 21 of 36...
Processing page 22 of 36...
Processing page 23 of 36...
Processing page 24 of 36...
Processing page 25 of 36...
Processing page 26 of 36...
Processing page 27 of 36...
Processing page 28 of 36...
Processing page 29 of 36...
Processing page 30 of 36...
Processing page 31 of 36...
Processing page 32 of 36...
Processing page 33 of 36...
Processing page 34 of 36...
Processing page 35 of 36...
Processing page 36 of 36...
T

In [7]:
ocr_text_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_statements_2025_ocr.txt"
)

with open(ocr_text_path, "w", encoding="utf-8") as text_file:
    text_file.write(all_pages_text)

print("OCR text saved successfully.")

OCR text saved successfully.


In [8]:
print(all_pages_text[:10000])



===== PAGE 1 =====

KABWE MUNICIPAL COUNCIL.
BI-ANNUAL FINANCLAL STATEMENTS FOR THE YEAR 2025

P O Box 80424
Civic Centre

KABWE



===== PAGE 2 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

TABLE OF CONTENTS

Report of the Council

Statement of Responsibilities for Bi-Annual Financial Statements

Independent Auditor’s Report

Statement of Cash Receipts and Payments

Statement of Comparison of Budget and Actual Amounts

Statement of Cash Receipts and Payments for Local Government Equalisation Fund
Statement of Cash Receipts and Payments for Constituency Development Fund
Statement of Cash Receipts and Payments for Sector Grant (Devolved Functions)

Statement of Cash Receipts and Payments for ZDSP Capital Grant

Summary of Significant Accounting Policies

Notes to the Financial Statements

16-19

20- 35


===== PAGE 3 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025 __

REPORT OF THE COUNCIL

The Council has the pleasure

In [9]:
print(all_pages_text[10000:20000])

s that are free from material
misstatement, whether due to fraud or error.

Nothing has come to the attention of the Council to indicate that the Kabwe Municipal Council

will not remain a going concern for at least twelve months from the date of this statement.

In the opinion of the Council, proper books of accounts were maintained to support preparation of
Financial Statements that present fairly the financial results of the Municipal Council for the bi-

financial year 2025

Signed on behalf of the Council on...............:ceeeeeeeeeeeneeee eens by;

Name: _ amy Crane — Name: ween AQ ty. MA op nLAQ
Signature \ a Signature........... a | seeeeeceeceesceeereeeeee
Position: Town Clerk Position: Director of Finance


===== PAGE 8 =====

REPUBLIC OF ZAMBIA
OFFICE OF THE AUDITOR GENERAL

INDEPENDENT AUDITOR’s REPORT


===== PAGE 9 =====

REPUBLIC OF ZAMBIA
OFFICE OF THE AUDITOR GENERAL

INDEPENDENT AUDITOR’s REPORT


===== PAGE 10 =====

REPUBLIC OF ZAMBIA
OFFICE OF THE AUDITOR GENERAL


In [10]:
# Display pages 7 to 36 one at a time
for page_number in range(7, 37):
    start_marker = f"===== PAGE {page_number} ====="
    end_marker = f"===== PAGE {page_number + 1} ====="

    start_index = all_pages_text.find(start_marker)

    if page_number < 36:
        end_index = all_pages_text.find(end_marker)
        page_text = all_pages_text[start_index:end_index]
    else:
        page_text = all_pages_text[start_index:]

    print(page_text)
    print("\n" + "=" * 80 + "\n")

===== PAGE 7 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

STATEMENT OF RESPONSIBILITIES FOR BI-ANNUAL FINANCIAL STATEMENTS
The Kabwe Municipal Council is responsible for preparing the bi-financial statements for the year
2025 which are free from material misstatement, whether due to fraud or error, and are prepared,
in all material respects, in accordance with the Cash Basis International Public Sector Accounting
Standard (IPSAS). In preparing the financial statements, the Council selected applicable policies
from Local Authorities Accounting Policies (LAAPs) of October 2019 and then applied them
consistently, making judgment and estimates that were reasonable and prudent.

The Council is also responsible for the maintenance of adequate accounting records and the
preparation and integrity of the annual financial statements and related information. The Auditor-
General has audited the financial statements, and his report is shown on pages 7 to 9.

The

In [11]:
search_terms = [
    "Cash Receipts",
    "Cash Payments",
    "Local Revenue",
    "Grants",
    "Equalisation Fund",
    "Constituency Development Fund",
    "CDF",
    "Revenue",
    "Expenditure",
    "Budget",
    "Actual",
    "Property, Plant",
    "Employees"
]

for term in search_terms:
    print(f"\n{'=' * 20} {term.upper()} {'=' * 20}")

    found = False

    for page_number in range(1, 37):
        start_marker = f"===== PAGE {page_number} ====="
        end_marker = f"===== PAGE {page_number + 1} ====="

        start_index = all_pages_text.find(start_marker)

        if page_number < 36:
            end_index = all_pages_text.find(end_marker)
            page_text = all_pages_text[start_index:end_index]
        else:
            page_text = all_pages_text[start_index:]

        if term.lower() in page_text.lower():
            print(f"\n--- Page {page_number} ---")
            print(page_text)
            found = True

    if not found:
        print("No matching text found.")


==================== CASH RECEIPTS ====================

--- Page 2 ---
===== PAGE 2 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

TABLE OF CONTENTS

Report of the Council

Statement of Responsibilities for Bi-Annual Financial Statements

Independent Auditor’s Report

Statement of Cash Receipts and Payments

Statement of Comparison of Budget and Actual Amounts

Statement of Cash Receipts and Payments for Local Government Equalisation Fund
Statement of Cash Receipts and Payments for Constituency Development Fund
Statement of Cash Receipts and Payments for Sector Grant (Devolved Functions)

Statement of Cash Receipts and Payments for ZDSP Capital Grant

Summary of Significant Accounting Policies

Notes to the Financial Statements

16-19

20- 35




--- Page 5 ---
===== PAGE 5 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

REPORT OF THE COUNCIL

The District also has two (2) elected Members of Parliament one for Kabwe C

In [12]:
import pandas as pd

financial_data = pd.DataFrame(columns=[
    "Financial Year",
    "Revenue Source",
    "Budget Amount",
    "Actual Amount",
    "Variance",
    "Fund Type",
    "Page Number",
    "Source Document"
])

financial_data

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document


In [13]:
financial_summary = pd.DataFrame([
    {
        "Financial Year": 2025,
        "Revenue Source": "Cash Receipts",
        "Budget Amount": None,
        "Actual Amount": 71746982,
        "Variance": None,
        "Fund Type": "General",
        "Page Number": 5,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Payments",
        "Budget Amount": None,
        "Actual Amount": 92307241,
        "Variance": None,
        "Fund Type": "General",
        "Page Number": 5,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Decrease in Cash and Cash Equivalents",
        "Budget Amount": None,
        "Actual Amount": -20560259,
        "Variance": None,
        "Fund Type": "General",
        "Page Number": 5,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Property, Plant and Equipment Acquired",
        "Budget Amount": None,
        "Actual Amount": 10453133.25,
        "Variance": None,
        "Fund Type": "Capital Assets",
        "Page Number": 6,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Employee Remuneration and Staff Welfare",
        "Budget Amount": None,
        "Actual Amount": 23866763.18,
        "Variance": None,
        "Fund Type": "Personnel",
        "Page Number": 6,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    }
])

financial_summary

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document
0,2025,Cash Receipts,None,71746982.00,None,General,5,Kabwe Municipal Council Bi-Annual Financial St...
1,2025,Payments,None,92307241.00,None,General,5,Kabwe Municipal Council Bi-Annual Financial St...
2,2025,Decrease in Cash and Cash Equivalents,None,-20560259.00,None,General,5,Kabwe Municipal Council Bi-Annual Financial St...
3,2025,"Property, Plant and Equipment Acquired",None,10453133.25,None,Capital Assets,6,Kabwe Municipal Council Bi-Annual Financial St...
4,2025,Employee Remuneration and Staff Welfare,None,23866763.18,None,Personnel,6,Kabwe Municipal Council Bi-Annual Financial St...


In [14]:
csv_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_summary_2025.csv"
)

financial_summary.to_csv(csv_path, index=False)

print("Financial summary saved successfully.")
print(csv_path)

Financial summary saved successfully.
C:\Users\Mutofwe\group6_administration\notebooks\financial_summary_2025.csv


In [15]:
# Read the saved CSV file
loaded_financial_summary = pd.read_csv(
    r"C:\Users\Mutofwe\group6_administration\notebooks\financial_summary_2025.csv"
)

# Display the data
display(loaded_financial_summary)

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document
0,2025,Cash Receipts,NaN,71746982.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
1,2025,Payments,NaN,92307241.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
2,2025,Decrease in Cash and Cash Equivalents,NaN,-20560259.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
3,2025,"Property, Plant and Equipment Acquired",NaN,10453133.25,NaN,Capital Assets,6,Kabwe Municipal Council Bi-Annual Financial St...
4,2025,Employee Remuneration and Staff Welfare,NaN,23866763.18,NaN,Personnel,6,Kabwe Municipal Council Bi-Annual Financial St...


In [16]:
print("Number of records:", len(loaded_financial_summary))
print("Number of columns:", len(loaded_financial_summary.columns))
print("\nColumn names:")
print(list(loaded_financial_summary.columns))

Number of records: 5
Number of columns: 8

Column names:
['Financial Year', 'Revenue Source', 'Budget Amount', 'Actual Amount', 'Variance', 'Fund Type', 'Page Number', 'Source Document']


In [17]:
print(loaded_financial_summary.isnull().sum())

Financial Year     0
Revenue Source     0
Budget Amount      5
Actual Amount      0
Variance           5
Fund Type          0
Page Number        0
Source Document    0
dtype: int64


In [18]:
amount_columns = [
    "Actual Amount"
]

loaded_financial_summary[amount_columns] = (
    loaded_financial_summary[amount_columns]
    .apply(pd.to_numeric, errors="coerce")
)

display(loaded_financial_summary)

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document
0,2025,Cash Receipts,NaN,71746982.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
1,2025,Payments,NaN,92307241.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
2,2025,Decrease in Cash and Cash Equivalents,NaN,-20560259.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
3,2025,"Property, Plant and Equipment Acquired",NaN,10453133.25,NaN,Capital Assets,6,Kabwe Municipal Council Bi-Annual Financial St...
4,2025,Employee Remuneration and Staff Welfare,NaN,23866763.18,NaN,Personnel,6,Kabwe Municipal Council Bi-Annual Financial St...


In [1]:
print("df_revenue" in globals())

False


In [2]:
revenue_records = []

In [4]:
import pandas as pd
import requests
import pdfplumber
import matplotlib.pyplot as plt
from IPython.display import display

In [5]:
print(pd.__version__)

3.0.5


In [6]:
revenue_records = []

In [7]:
df_revenue = pd.DataFrame(revenue_records)

In [8]:
print("df_revenue" in globals())
print("Number of records:", len(df_revenue))
display(df_revenue.head())

True
Number of records: 0


""


In [9]:
print("df_revenue" in globals())
print("Number of records:", len(df_revenue))
display(df_revenue.head())

True
Number of records: 0


""


In [10]:
print("Number of revenue records:", len(revenue_records))
print(revenue_records)

Number of revenue records: 0
[]


In [11]:
revenue_records.append({
    "page": page_number,
    "revenue_category": category,
    "revenue_code": code,
    "revenue_description": description,
    "budget_2025": budget_2025,
    "revised_budget_2026": revised_budget_2026,
    "estimate_2027": estimate_2027
})

NameError: name 'page_number' is not defined

In [12]:
revenue_records.append({
    "page": page_number,
    ...
})

SyntaxError: ':' expected after dictionary key (1780305571.py, line 3)

In [13]:
df_revenue = pd.DataFrame(revenue_records)

print("Number of records:", len(df_revenue))
display(df_revenue.head())

Number of records: 0


""


In [14]:
print("pdf" in globals())

False


In [15]:
import pdfplumber

pdf_path = r"C:\Users\Mutofwe\group6_administration\notebooks\2025-KABWE-M-COUNCIL-OBB.pdf"

pdf = pdfplumber.open(pdf_path)

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mutofwe\\group6_administration\\notebooks\\2025-KABWE-M-COUNCIL-OBB.pdf'

In [16]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(BytesIO(budget_response.content))

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

NameError: name 'budget_response' is not defined

In [17]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

budget_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2026/04/2026-Approved-OBB-Budget.pdf"

budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("File size:", len(budget_response.content), "bytes")

Status code: 200
File size: 1936931 bytes


In [18]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(BytesIO(budget_response.content))

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

PDF opened successfully
Number of pages: 59


In [19]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

budget_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf"

budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("File size:", len(budget_response.content), "bytes")

Status code: 200
File size: 792438 bytes


In [20]:
budget_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf"

In [21]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(BytesIO(budget_response.content))

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

PDF opened successfully
Number of pages: 54


In [22]:
page2 = pdf.pages[1]

table = page2.extract_table({
    "vertical_strategy": "text",
    "horizontal_strategy": "text",
    "intersection_tolerance": 5
})

for row in table:
    print(row)

['Pag', 'e 2', 'OUTPUT BASED ANNUAL B', 'UDGET', '', '']
['', '', '', '', '', '']
['HE', 'A 920', 'KABWE MUNICIPAL COUNCIL', '', '', '']
['D', '5', '', '', '', '']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', 'CODE', 'REVENUE DESCRIPTION', 'APPROVED', 'REVISED', 'BUDGET']
['', '', 'B', 'UDGET 2025\nB', 'UDGET 2026 E', 'STIMATE 2']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', '01', 'Local taxes/rates', '', '', '']
['', '', '', '', '', '']
['', '001', 'Residential', '4,453,818', '4,453,818', '4,453,818']
['', '', '', '', '', '']
['', '002', 'Commercial', '5,536,418', '5,536,418', '5,536,418']
['', '', '', '', '', '']
['', '003', 'Industrial', '2,635,280', '2,635,280', '2,635,280']
['', '', '', '', '', '']
['', '004', 'Hospitality', '572,837', '630,121', '693,133']
['', '', '', '', '', '']
['', '', 'SubItem Total', '13,198,353', '13,255,637', '13,318,649']
['', '001', 'Personal levy', '450,000', '495,000', '220,000']
['', '', '', '', '', '']
['', '', 'SubItem Total', '

In [23]:
import re
import pandas as pd

records = []
current_category = None

for item in budget_rows:
    page_number = item["page"]
    row = item["row"]

    if not row:
        continue

    # Clean cells
    row = [cell.strip() if cell else "" for cell in row]

    # Find category codes such as 01, 02, 03 ... 08
    category_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{2}", cell):
            category_index = i
            break

    if category_index is not None:
        # Category name is usually immediately after the 2-digit code
        if category_index + 1 < len(row):
            possible_category = row[category_index + 1].strip()

            if possible_category:
                current_category = possible_category

        continue

    # Find 3-digit revenue codes such as 001, 002, 099
    code_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{3}", cell):
            code_index = i
            break

    if code_index is None:
        continue

    # Ignore subtotal rows
    if any("SubItem Total" in cell for cell in row):
        continue

    # Description is normally the cell immediately after the code
    description_parts = []

    for cell in row[code_index + 1:]:
        if cell:
            description_parts.append(cell)

    if not description_parts:
        continue

    # First non-numeric cells form the description
    description = " ".join(description_parts[:2]).strip()

    # Locate numeric budget values
    numeric_values = []

    for cell in row[code_index + 1:]:
        cleaned = cell.replace(",", "").strip()

        if re.fullmatch(r"-?\d+(?:\.\d+)?", cleaned):
            numeric_values.append(float(cleaned))

    # We need three financial values
    if len(numeric_values) >= 3:
        records.append({
            "page": page_number,
            "revenue_category": current_category,
            "revenue_code": row[code_index],
            "revenue_description": description,
            "approved_budget_2025": numeric_values[-3],
            "revised_budget_2026": numeric_values[-2],
            "estimate_2027": numeric_values[-1]
        })

df_revenue = pd.DataFrame(records)

print("Number of records:", len(df_revenue))
display(df_revenue)

NameError: name 'budget_rows' is not defined

In [24]:
budget_rows = []

for page_number in range(2, 6):
    page = pdf.pages[page_number - 1]

    table = page.extract_table({
        "vertical_strategy": "text",
        "horizontal_strategy": "text",
        "intersection_tolerance": 5
    })

    if table:
        for row in table:
            budget_rows.append({
                "page": page_number,
                "row": row
            })

print("Total extracted rows:", len(budget_rows))

Total extracted rows: 269


In [25]:
import re
import pandas as pd

records = []
current_category = None

for item in budget_rows:
    page_number = item["page"]
    row = item["row"]

    if not row:
        continue

    row = [cell.strip() if cell else "" for cell in row]

    # Find 2-digit category code: 01, 02, ..., 08
    category_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{2}", cell):
            category_index = i
            break

    if category_index is not None:
        if category_index + 1 < len(row):
            category_name = row[category_index + 1].strip()

            if category_name:
                current_category = category_name

        continue

    # Find 3-digit revenue code
    code_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{3}", cell):
            code_index = i
            break

    if code_index is None:
        continue

    # Skip subtotal rows
    if any("SubItem Total" in cell for cell in row):
        continue

    # Description
    description_parts = []

    for cell in row[code_index + 1:]:
        if cell:
            cleaned = cell.replace(",", "").strip()

            if not re.fullmatch(r"-?\d+(?:\.\d+)?", cleaned):
                description_parts.append(cell)

    description = " ".join(description_parts).strip()

    # Numeric values
    numeric_values = []

    for cell in row[code_index + 1:]:
        cleaned = cell.replace(",", "").strip()

        if re.fullmatch(r"-?\d+(?:\.\d+)?", cleaned):
            numeric_values.append(float(cleaned))

    if len(numeric_values) >= 3:
        records.append({
            "page": page_number,
            "revenue_category": current_category,
            "revenue_code": row[code_index],
            "revenue_description": description,
            "approved_budget_2025": numeric_values[-3],
            "revised_budget_2026": numeric_values[-2],
            "estimate_2027": numeric_values[-1]
        })

df_revenue = pd.DataFrame(records)

print("Number of records:", len(df_revenue))
display(df_revenue)

Number of records: 80


,page,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
...,...,...,...,...,...,...,...
75,5,Nationa,002,Roads Gr ant,3.0,200587.0,3200587.0
76,5,Nationa,003,Health Gr ant,3.0,649035.0,3649035.0
77,5,Nationa,004,Local Go vernment Equ alisation Fund,2.0,3857381.0,23857381.0
78,5,Nationa,005,Grants in lieu of Rates,1.0,200000.0,1200000.0


In [26]:
print(df_revenue["revenue_category"].unique())
print()
print(df_revenue["revenue_category"].value_counts())

<StringArray>
['Local taxes/rates',  'Fees and Charges',          'Licenses',
            'Levies',           'Permits',           'Charges',
          'Other In',           'Nationa']
Length: 8, dtype: str

revenue_category
Fees and Charges     39
Levies               11
Charges               9
Nationa               6
Local taxes/rates     5
Permits               5
Licenses              4
Other In              1
Name: count, dtype: int64


In [27]:
df_revenue["revenue_category"] = df_revenue["revenue_category"].replace({
    "Other In": "Other Incomes",
    "Nationa": "National Support (Grants)"
})

print(df_revenue["revenue_category"].unique())

<StringArray>
[        'Local taxes/rates',          'Fees and Charges',
                  'Licenses',                    'Levies',
                   'Permits',                   'Charges',
             'Other Incomes', 'National Support (Grants)']
Length: 8, dtype: str


In [28]:
print("Total records:", len(df_revenue))

print("\nExtracted 2025 Approved Budget total:")
print(df_revenue["approved_budget_2025"].sum())

print("\nOfficial Grand Total:")
print(181079550)

Total records: 80

Extracted 2025 Approved Budget total:
44662836.0

Official Grand Total:
181079550


In [29]:
df_revenue.groupby("revenue_category")["approved_budget_2025"].sum()

revenue_category
Charges                       9586050.0
Fees and Charges             13038015.0
Levies                        1720900.0
Licenses                       832000.0
Local taxes/rates            13648353.0
National Support (Grants)          17.0
Other Incomes                       1.0
Permits                       5837500.0
Name: approved_budget_2025, dtype: float64

In [30]:
df_revenue[
    df_revenue["revenue_category"] == "National Support (Grants)"
]

,page,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027
74,5,National Support (Grants),001,Constitue ncy Develop ment Fund,7.0,2116301.0,72116301.0
75,5,National Support (Grants),002,Roads Gr ant,3.0,200587.0,3200587.0
76,5,National Support (Grants),003,Health Gr ant,3.0,649035.0,3649035.0
77,5,National Support (Grants),004,Local Go vernment Equ alisation Fund,2.0,3857381.0,23857381.0
78,5,National Support (Grants),005,Grants in lieu of Rates,1.0,200000.0,1200000.0
79,5,National Support (Grants),099,Other Gr ants,1.0,527645.0,10797645.0


In [31]:
# Fix split numbers in National Support (Grants)

grant_mask = df_revenue["revenue_category"] == "National Support (Grants)"

df_revenue.loc[grant_mask, "approved_budget_2025"] = [
    72116301,
    3200587,
    3649035,
    23857381,
    1200000,
    29494551
]

print(df_revenue[grant_mask])

    page           revenue_category revenue_code  \
74     5  National Support (Grants)          001   
75     5  National Support (Grants)          002   
76     5  National Support (Grants)          003   
77     5  National Support (Grants)          004   
78     5  National Support (Grants)          005   
79     5  National Support (Grants)          099   

                     revenue_description  approved_budget_2025  \
74       Constitue ncy Develop ment Fund            72116301.0   
75                          Roads Gr ant             3200587.0   
76                         Health Gr ant             3649035.0   
77  Local Go vernment Equ alisation Fund            23857381.0   
78               Grants in lieu of Rates             1200000.0   
79                         Other Gr ants            29494551.0   

    revised_budget_2026  estimate_2027  
74            2116301.0     72116301.0  
75             200587.0      3200587.0  
76             649035.0      3649035.0  
77      

In [32]:
print("Total records:", len(df_revenue))

print("\nCorrected 2025 Approved Budget total:")
print(df_revenue["approved_budget_2025"].sum())

print("\nOfficial Grand Total:")
print(181079550)

Total records: 80

Corrected 2025 Approved Budget total:
178180674.0

Official Grand Total:
181079550


In [33]:
other_income_mask = df_revenue["revenue_category"] == "Other Incomes"

df_revenue.loc[other_income_mask, "approved_budget_2025"] = 2898877

print(df_revenue[other_income_mask])

    page revenue_category revenue_code revenue_description  \
73     5    Other Incomes          099       Other Inc ome   

    approved_budget_2025  revised_budget_2026  estimate_2027  
73             2898877.0             500000.0      1500000.0  


In [34]:
print("Corrected total:")
print(df_revenue["approved_budget_2025"].sum())

print("Official total:")
print(181079550)

print("Difference:")
print(181079550 - df_revenue["approved_budget_2025"].sum())

Corrected total:
181079550.0
Official total:
181079550
Difference:
0.0


In [35]:
category_totals = (
    df_revenue
    .groupby("revenue_category")["approved_budget_2025"]
    .sum()
    .sort_values(ascending=False)
)

display(category_totals)

revenue_category
National Support (Grants)    133517855.0
Local taxes/rates             13648353.0
Fees and Charges              13038015.0
Charges                        9586050.0
Permits                        5837500.0
Other Incomes                  2898877.0
Levies                         1720900.0
Licenses                        832000.0
Name: approved_budget_2025, dtype: float64

In [36]:
print("Total of category totals:")
print(category_totals.sum())

Total of category totals:
181079550.0


In [37]:
# Make a clean copy of the validated dataset
final_2025_budget = df_revenue.copy()

# Clean text fields
final_2025_budget["revenue_category"] = (
    final_2025_budget["revenue_category"]
    .astype(str)
    .str.strip()
)

final_2025_budget["revenue_description"] = (
    final_2025_budget["revenue_description"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Make sure codes remain 3 digits
final_2025_budget["revenue_code"] = (
    final_2025_budget["revenue_code"]
    .astype(str)
    .str.zfill(3)
)

# Add the budget year
final_2025_budget["financial_year"] = 2025

# Reorder columns
final_2025_budget = final_2025_budget[
    [
        "financial_year",
        "revenue_category",
        "revenue_code",
        "revenue_description",
        "approved_budget_2025",
        "revised_budget_2026",
        "estimate_2027",
        "page"
    ]
]

display(final_2025_budget.head())
print("Number of records:", len(final_2025_budget))

,financial_year,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027,page
0,2025,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0,2
1,2025,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0,2
2,2025,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0,2
3,2025,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0,2
4,2025,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0,2


Number of records: 80


In [38]:
print(final_2025_budget.isnull().sum())

financial_year          0
revenue_category        0
revenue_code            0
revenue_description     0
approved_budget_2025    0
revised_budget_2026     0
estimate_2027           0
page                    0
dtype: int64


In [39]:
print("Duplicate rows:", final_2025_budget.duplicated().sum())

Duplicate rows: 0


In [40]:
csv_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv"

final_2025_budget.to_csv(
    csv_path,
    sep="|",
    index=False
)

print("CSV saved successfully!")
print(csv_path)

CSV saved successfully!
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv


In [41]:
check = pd.read_csv(csv_path, sep="|")

print("Rows:", len(check))
print("Columns:", list(check.columns))

display(check.head())

Rows: 80
Columns: ['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']


,financial_year,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027,page
0,2025,Local taxes/rates,1,Residential,4453818.0,4453818.0,4453818.0,2
1,2025,Local taxes/rates,2,Commercial,5536418.0,5536418.0,5536418.0,2
2,2025,Local taxes/rates,3,Industrial,2635280.0,2635280.0,2635280.0,2
3,2025,Local taxes/rates,4,Hospitality,572837.0,630121.0,693133.0,2
4,2025,Local taxes/rates,1,Personal levy,450000.0,495000.0,220000.0,2


In [42]:
lgef_data = [
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "1st Funding",
        "amount": 1843986,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "2nd Funding",
        "amount": 1764901,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "3rd Funding",
        "amount": 1832551,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "4th Funding",
        "amount": 1730867,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "5th Funding",
        "amount": 1430192,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Operational Expenditure",
        "description": "Personal emoluments, salaries and wages",
        "amount": 8602496,
        "purpose": "Operational expenditure",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Capital Expenditure",
        "description": "Lukanga Bus Station",
        "amount": 6146594,
        "purpose": "Capital expenditure",
        "page": 25
    }
]

lgef_df = pd.DataFrame(lgef_data)

display(lgef_df)

,financial_year,fund_type,transaction_type,description,amount,purpose,page
0,2025,LGEF,Funding,1st Funding,1843986,LGEF Funding,25
1,2025,LGEF,Funding,2nd Funding,1764901,LGEF Funding,25
2,2025,LGEF,Funding,3rd Funding,1832551,LGEF Funding,25
3,2025,LGEF,Funding,4th Funding,1730867,LGEF Funding,25
4,2025,LGEF,Funding,5th Funding,1430192,LGEF Funding,25
5,2025,LGEF,Operational Expenditure,"Personal emoluments, salaries and wages",8602496,Operational expenditure,25
6,2025,LGEF,Capital Expenditure,Lukanga Bus Station,6146594,Capital expenditure,25


In [43]:
# Check for duplicate rows
print("Duplicate rows:", lgef_df.duplicated().sum())

# Check for missing values
print("\nMissing values:")
print(lgef_df.isnull().sum())

# Verify LGEF funding total
funding_total = lgef_df[
    lgef_df["transaction_type"] == "Funding"
]["amount"].sum()

print("\nLGEF funding total:", funding_total)
print("Official funding total:", 8602496)
print("Difference:", 8602496 - funding_total)

Duplicate rows: 0

Missing values:
financial_year      0
fund_type           0
transaction_type    0
description         0
amount              0
purpose             0
page                0
dtype: int64

LGEF funding total: 8602497
Official funding total: 8602496
Difference: -1


In [44]:
funding_rows = lgef_df[
    lgef_df["transaction_type"] == "Funding"
].copy()

display(funding_rows)

print("\nIndividual funding amounts:")
for _, row in funding_rows.iterrows():
    print(row["description"], "=", row["amount"])

print("\nCalculated total:", funding_rows["amount"].sum())
print("Official total:", 8602496)

,financial_year,fund_type,transaction_type,description,amount,purpose,page
0,2025,LGEF,Funding,1st Funding,1843986,LGEF Funding,25
1,2025,LGEF,Funding,2nd Funding,1764901,LGEF Funding,25
2,2025,LGEF,Funding,3rd Funding,1832551,LGEF Funding,25
3,2025,LGEF,Funding,4th Funding,1730867,LGEF Funding,25
4,2025,LGEF,Funding,5th Funding,1430192,LGEF Funding,25



Individual funding amounts:
1st Funding = 1843986
2nd Funding = 1764901
3rd Funding = 1832551
4th Funding = 1730867
5th Funding = 1430192

Calculated total: 8602497
Official total: 8602496


In [45]:
print(lgef_df.to_string(index=False))

 financial_year fund_type        transaction_type                             description  amount                 purpose  page
           2025      LGEF                 Funding                             1st Funding 1843986            LGEF Funding    25
           2025      LGEF                 Funding                             2nd Funding 1764901            LGEF Funding    25
           2025      LGEF                 Funding                             3rd Funding 1832551            LGEF Funding    25
           2025      LGEF                 Funding                             4th Funding 1730867            LGEF Funding    25
           2025      LGEF                 Funding                             5th Funding 1430192            LGEF Funding    25
           2025      LGEF Operational Expenditure Personal emoluments, salaries and wages 8602496 Operational expenditure    25
           2025      LGEF     Capital Expenditure                     Lukanga Bus Station 6146594     Ca

In [46]:
print("Calculated funding total:", funding_total)
print("Official funding total:", 8602496)
print("Difference (official - calculated):", 8602496 - funding_total)

Calculated funding total: 8602497
Official funding total: 8602496
Difference (official - calculated): -1


In [47]:
csv_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv"

lgef_df.to_csv(
    csv_path,
    sep="|",
    index=False
)

print("LGEF dataset saved successfully:")
print(csv_path)

LGEF dataset saved successfully:
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv


In [48]:
import os

print("File exists:", os.path.exists(csv_path))
print("File size:", os.path.getsize(csv_path), "bytes")

File exists: True
File size: 542 bytes


In [49]:
local_revenue_data = [
    {
        "financial_year": 2025,
        "revenue_type": "Local taxes",
        "amount": 6663177,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Fees and Charges",
        "amount": 12062057,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Licences",
        "amount": 347920,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Levies",
        "amount": 1209808,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Permits",
        "amount": 2726912,
        "source_page": 11
    }
]

local_revenue_df = pd.DataFrame(local_revenue_data)

display(local_revenue_df)

,financial_year,revenue_type,amount,source_page
0,2025,Local taxes,6663177,11
1,2025,Fees and Charges,12062057,11
2,2025,Licences,347920,11
3,2025,Levies,1209808,11
4,2025,Permits,2726912,11


In [50]:
# Check for duplicate rows
print("Duplicate rows:", local_revenue_df.duplicated().sum())

# Check for missing values
print("\nMissing values:")
print(local_revenue_df.isnull().sum())

# Calculate total actual local revenue
local_revenue_total = local_revenue_df["amount"].sum()

print("\nCalculated local revenue total:", local_revenue_total)

Duplicate rows: 0

Missing values:
financial_year    0
revenue_type      0
amount            0
source_page       0
dtype: int64

Calculated local revenue total: 23009874


In [51]:
csv_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv"

local_revenue_df.to_csv(
    csv_path,
    sep="|",
    index=False
)

print("Local revenue dataset saved successfully:")
print(csv_path)

Local revenue dataset saved successfully:
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv


In [52]:
import os

print("File exists:", os.path.exists(csv_path))
print("File size:", os.path.getsize(csv_path), "bytes")

File exists: True
File size: 186 bytes


In [53]:
import os

files_to_check = [
    r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv",
    r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv",
    r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv"
]

for file in files_to_check:
    print(os.path.basename(file), "→", os.path.exists(file))

db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv → True
db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv → True
db-unza26-csc4792-kabwe-local-revenue-2025.csv → True


In [54]:
import pandas as pd

approved_budget_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv"
lgef_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv"
local_revenue_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv"

approved_budget_check = pd.read_csv(approved_budget_path, sep="|")
lgef_check = pd.read_csv(lgef_path, sep="|")
local_revenue_check = pd.read_csv(local_revenue_path, sep="|")

print("APPROVED BUDGET")
print(approved_budget_check.shape)
print(approved_budget_check.columns.tolist())

print("\nLGEF UTILISATION")
print(lgef_check.shape)
print(lgef_check.columns.tolist())

print("\nLOCAL REVENUE")
print(local_revenue_check.shape)
print(local_revenue_check.columns.tolist())

APPROVED BUDGET
(80, 8)
['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']

LGEF UTILISATION
(7, 7)
['financial_year', 'fund_type', 'transaction_type', 'description', 'amount', 'purpose', 'page']

LOCAL REVENUE
(5, 4)
['financial_year', 'revenue_type', 'amount', 'source_page']


In [55]:
print("===== APPROVED BUDGET =====")
display(approved_budget_check)

print("===== LGEF UTILISATION =====")
display(lgef_check)

print("===== LOCAL REVENUE =====")
display(local_revenue_check)

===== APPROVED BUDGET =====


,financial_year,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027,page
0,2025,Local taxes/rates,1,Residential,4453818.0,4453818.0,4453818.0,2
1,2025,Local taxes/rates,2,Commercial,5536418.0,5536418.0,5536418.0,2
2,2025,Local taxes/rates,3,Industrial,2635280.0,2635280.0,2635280.0,2
3,2025,Local taxes/rates,4,Hospitality,572837.0,630121.0,693133.0,2
4,2025,Local taxes/rates,1,Personal levy,450000.0,495000.0,220000.0,2
...,...,...,...,...,...,...,...,...
75,2025,National Support (Grants),2,Roads Gr ant,3200587.0,200587.0,3200587.0,5
76,2025,National Support (Grants),3,Health Gr ant,3649035.0,649035.0,3649035.0,5
77,2025,National Support (Grants),4,Local Go vernment Equ alisation Fund,23857381.0,3857381.0,23857381.0,5
78,2025,National Support (Grants),5,Grants in lieu of Rates,1200000.0,200000.0,1200000.0,5


===== LGEF UTILISATION =====


,financial_year,fund_type,transaction_type,description,amount,purpose,page
0,2025,LGEF,Funding,1st Funding,1843986,LGEF Funding,25
1,2025,LGEF,Funding,2nd Funding,1764901,LGEF Funding,25
2,2025,LGEF,Funding,3rd Funding,1832551,LGEF Funding,25
3,2025,LGEF,Funding,4th Funding,1730867,LGEF Funding,25
4,2025,LGEF,Funding,5th Funding,1430192,LGEF Funding,25
5,2025,LGEF,Operational Expenditure,"Personal emoluments, salaries and wages",8602496,Operational expenditure,25
6,2025,LGEF,Capital Expenditure,Lukanga Bus Station,6146594,Capital expenditure,25


===== LOCAL REVENUE =====


,financial_year,revenue_type,amount,source_page
0,2025,Local taxes,6663177,11
1,2025,Fees and Charges,12062057,11
2,2025,Licences,347920,11
3,2025,Levies,1209808,11
4,2025,Permits,2726912,11


In [56]:
# Search the extracted 2025 financial statement text
with open(
    r"C:\Users\Mutofwe\group6_administration\notebooks\financial_statements_2025_ocr.txt",
    "r",
    encoding="utf-8"
) as f:
    financial_text = f.read()

# Look for individual local revenue streams
keywords = [
    "market fees",
    "local taxes",
    "fees and charges",
    "levies",
    "licences",
    "permits",
    "market"
]

for keyword in keywords:
    print("\n" + "=" * 70)
    print("SEARCH:", keyword)
    print("=" * 70)

    matches = [
        line for line in financial_text.splitlines()
        if keyword.lower() in line.lower()
    ]

    for line in matches[:20]:
        print(line)


SEARCH: market fees
Market Fees

SEARCH: local taxes
Local taxes Zi 6,663,177
Local taxes 6,824,177 0 6,824,177 6,663,177 98 160,999 2
a. Local Taxes
to the Constitution and the Business Regulatory Act of 2014, a system of local taxes
2. Local Taxes

SEARCH: fees and charges
make regulations, imposition of levies, fees and charges and to formulate local policies to promote,
Fees and Charges 3 12,062,057
Fees and Charges 11,187,033 11,187,033 12,062,057 108 - 875,024 - 8
b. Fees and Charges
3. Fees and Charges
The Council generated cash receipts in form of fees and charges arising from offering
Fees and charges 5,156,007 -
a) Fees and Charges
Other Fees and Charges

SEARCH: levies
make regulations, imposition of levies, fees and charges and to formulate local policies to promote,
Levies 5 1,209,808
Levies 860,450 - 860,450 1,209,808 141 (349,358) - 41
which Local Authorities can raise by passing by-laws imposing levies on:
5. Levies
The Council generated cash receipts by charging levie

In [57]:
detailed_local_revenue = [
    # Local Taxes
    {"financial_year": 2025, "revenue_category": "Local Taxes", "revenue_stream": "Residential Rates", "actual_amount": 1948368, "source_page": 21},
    {"financial_year": 2025, "revenue_category": "Local Taxes", "revenue_stream": "Industrial / Commercial Rates", "actual_amount": 4492657, "source_page": 21},
    {"financial_year": 2025, "revenue_category": "Local Taxes", "revenue_stream": "Personal Levy", "actual_amount": 222153, "source_page": 21},

    # Fees and Charges
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Consent Fees", "actual_amount": 6000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Survey Fees", "actual_amount": 8000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Building Inspection Fees", "actual_amount": 249300, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Plan Scrutiny Fees", "actual_amount": 553935, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Change of Premise Use", "actual_amount": 59162, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Container / Ntemba Fees", "actual_amount": 16084, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Rentals / Lease of Council Properties", "actual_amount": 688925, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Application Form Fees", "actual_amount": 501500, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Search Fees", "actual_amount": 2150, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Notice Board Adverts", "actual_amount": 247111, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Market Fees", "actual_amount": 89016, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Parking Fees", "actual_amount": 262033, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Bus Station Fees", "actual_amount": 11770, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Affidavit Fees", "actual_amount": 6000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Hire of Hall", "actual_amount": 1900, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Hire of Stadia", "actual_amount": 1120, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Body Transfer / Inspection Settlement", "actual_amount": 1800, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Boundary Location", "actual_amount": 349850, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Refuse Disposal Fees", "actual_amount": 1000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Library Fees", "actual_amount": 24500, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Notice of Marriage", "actual_amount": 100, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Abattoir / Meat Inspection Fees", "actual_amount": 17150, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Registration of Clubs and Societies", "actual_amount": 24146, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Farm Produce", "actual_amount": 344437, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Communication Mast Levy", "actual_amount": 51900, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Illegal Parking Fees", "actual_amount": 123030, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Sale of Parks", "actual_amount": 200, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Billboard and Banner", "actual_amount": 484813, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Lease of Council Transport", "actual_amount": 559244, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Penalties", "actual_amount": 75000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Ablution Fees", "actual_amount": 1200, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Extracts of Minutes", "actual_amount": 99600, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Recommendation Fees", "actual_amount": 234750, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Medical Fees", "actual_amount": 59280, "source_page": 22},

    # Land Development Charges
    {"financial_year": 2025, "revenue_category": "Land Development Charges", "revenue_stream": "Premium Plots - Residential", "actual_amount": 5350800, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Land Development Charges", "revenue_stream": "Premium Plots - Commercial", "actual_amount": 1365500, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Land Development Charges", "revenue_stream": "Other", "actual_amount": 189750, "source_page": 23},

    # Licences
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Liquor Licence", "actual_amount": 221180, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Firearm and Ammunition", "actual_amount": 15000, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Dog Licence", "actual_amount": 6200, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Petroleum", "actual_amount": 48370, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Occupancy", "actual_amount": 39620, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Other Licence", "actual_amount": 17550, "source_page": 23},

    # Levies
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Bird Levy", "actual_amount": 54080, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Business Levy", "actual_amount": 807848, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Pole Levy", "actual_amount": 3442, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Telecommunication Mast", "actual_amount": 344437, "source_page": 24},

    # Permits
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Health Permit", "actual_amount": 1136180, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Burial Permits and Grave Sites", "actual_amount": 150875, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Fire Certificate", "actual_amount": 1418357, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Extension of Business Hours Permits", "actual_amount": 1250, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Public Permits", "actual_amount": 2750, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Other Permits", "actual_amount": 17500, "source_page": 24},
]

detailed_local_revenue_df = pd.DataFrame(detailed_local_revenue)

display(detailed_local_revenue_df)
print("Number of records:", len(detailed_local_revenue_df))

,financial_year,revenue_category,revenue_stream,actual_amount,source_page
0,2025,Local Taxes,Residential Rates,1948368,21
1,2025,Local Taxes,Industrial / Commercial Rates,4492657,21
2,2025,Local Taxes,Personal Levy,222153,21
3,2025,Fees and Charges,Consent Fees,6000,22
4,2025,Fees and Charges,Survey Fees,8000,22
5,2025,Fees and Charges,Building Inspection Fees,249300,22
6,2025,Fees and Charges,Plan Scrutiny Fees,553935,22
7,2025,Fees and Charges,Change of Premise Use,59162,22
8,2025,Fees and Charges,Container / Ntemba Fees,16084,22
9,2025,Fees and Charges,Rentals / Lease of Council Properties,688925,22


Number of records: 56


In [58]:
for category, official in official_totals.items():
    calculated = calculated_totals.get(category, 0)
    difference = official - calculated

    print(
        f"{category}: "
        f"Official = K{official:,}, "
        f"Calculated = K{calculated:,}, "
        f"Difference = K{difference:,}"
    )

NameError: name 'official_totals' is not defined

In [59]:
# Official totals taken from the 2025 financial statement
official_totals = {
    "Local Taxes": 6663177,
    "Fees and Charges": 5156007,
    "Land Development Charges": 6906050,
    "Licences": 347920,
    "Levies": 1209808,
    "Permits": 2726912
}

# Calculate totals from the detailed_local_revenue dataframe
calculated_totals = (
    detailed_local_revenue
    .groupby("revenue_category")["actual_amount"]
    .sum()
    .to_dict()
)

# Compare official totals against calculated totals
for category, official in official_totals.items():
    calculated = calculated_totals.get(category, 0)
    difference = official - calculated

    print(
        f"{category}: "
        f"Official = K{official:,.0f}, "
        f"Calculated = K{calculated:,.0f}, "
        f"Difference (official - calculated) = K{difference:,.0f}"
    )

AttributeError: 'list' object has no attribute 'groupby'

In [60]:
import pandas as pd

# Convert the detailed revenue list into a DataFrame
detailed_local_revenue = pd.DataFrame(detailed_local_revenue)

# Check the structure
print("Rows:", len(detailed_local_revenue))
print("Columns:", detailed_local_revenue.columns.tolist())

detailed_local_revenue.head()

Rows: 56
Columns: ['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page']


,financial_year,revenue_category,revenue_stream,actual_amount,source_page
0,2025,Local Taxes,Residential Rates,1948368,21
1,2025,Local Taxes,Industrial / Commercial Rates,4492657,21
2,2025,Local Taxes,Personal Levy,222153,21
3,2025,Fees and Charges,Consent Fees,6000,22
4,2025,Fees and Charges,Survey Fees,8000,22


In [61]:
# Official totals from the 2025 financial statement
official_totals = {
    "Local Taxes": 6663177,
    "Fees and Charges": 5156007,
    "Land Development Charges": 6906050,
    "Licences": 347920,
    "Levies": 1209808,
    "Permits": 2726912
}

# Calculate totals from the detailed records
calculated_totals = (
    detailed_local_revenue
    .groupby("revenue_category")["actual_amount"]
    .sum()
    .to_dict()
)

# Compare official totals with calculated totals
for category, official in official_totals.items():
    calculated = calculated_totals.get(category, 0)
    difference = official - calculated

    print(
        f"{category}: "
        f"Official = K{official:,.0f}, "
        f"Calculated = K{calculated:,.0f}, "
        f"Difference (official - calculated) = K{difference:,.0f}"
    )

Local Taxes: Official = K6,663,177, Calculated = K6,663,178, Difference (official - calculated) = K-1
Fees and Charges: Official = K5,156,007, Calculated = K5,156,006, Difference (official - calculated) = K1
Land Development Charges: Official = K6,906,050, Calculated = K6,906,050, Difference (official - calculated) = K0
Licences: Official = K347,920, Calculated = K347,920, Difference (official - calculated) = K0
Levies: Official = K1,209,808, Calculated = K1,209,807, Difference (official - calculated) = K1
Permits: Official = K2,726,912, Calculated = K2,726,912, Difference (official - calculated) = K0


In [62]:
# Add validation status to the dataset
detailed_local_revenue["source_validation_note"] = (
    "Detailed source amount; category total may differ from published total by K1 due to source rounding/reporting discrepancy."
)

# Display final dataset structure
print("Number of records:", len(detailed_local_revenue))
print("\nColumns:")
print(detailed_local_revenue.columns.tolist())

print("\nMissing values:")
print(detailed_local_revenue.isnull().sum())

# Check duplicates
print("\nDuplicate rows:", detailed_local_revenue.duplicated().sum())

# Export detailed local revenue dataset
local_revenue_file = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv"
)

detailed_local_revenue.to_csv(
    local_revenue_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("\nExported successfully:")
print(local_revenue_file)

Number of records: 56

Columns:
['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page', 'source_validation_note']

Missing values:
financial_year            0
revenue_category          0
revenue_stream            0
actual_amount             0
source_page               0
source_validation_note    0
dtype: int64

Duplicate rows: 0

Exported successfully:
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv


In [63]:
import os
import pandas as pd

# ---------------------------------------------------------
# FINAL DATASET CHECK
# ---------------------------------------------------------

files_to_check = {
    "Approved Budget":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv",

    "LGEF Utilisation":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv",

    "Detailed Local Revenue":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv"
}

for name, path in files_to_check.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("File exists:", os.path.exists(path))

    if os.path.exists(path):

        print("File size:", os.path.getsize(path), "bytes")

        df = pd.read_csv(path, sep="|")

        print("Rows:", len(df))
        print("Columns:", len(df.columns))
        print("Column names:", df.columns.tolist())

        print("Missing values:")
        print(df.isnull().sum().sum())

        print("Duplicate rows:", df.duplicated().sum())

        print("Separator check: PASSED")


Approved Budget
File exists: True
File size: 6106 bytes
Rows: 80
Columns: 8
Column names: ['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']
Missing values:
0
Duplicate rows: 0
Separator check: PASSED

LGEF Utilisation
File exists: True
File size: 542 bytes
Rows: 7
Columns: 7
Column names: ['financial_year', 'fund_type', 'transaction_type', 'description', 'amount', 'purpose', 'page']
Missing values:
0
Duplicate rows: 0
Separator check: PASSED

Detailed Local Revenue
File exists: True
File size: 9715 bytes
Rows: 56
Columns: 6
Column names: ['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page', 'source_validation_note']
Missing values:
0
Duplicate rows: 0
Separator check: PASSED


In [64]:
import os
import pandas as pd

files_to_check = {
    "Approved Budget":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv",

    "LGEF Utilisation":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv",

    "Detailed Local Revenue":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv"
}

for name, path in files_to_check.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("File exists:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path, sep="|")

        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print("Missing values:", df.isnull().sum().sum())
        print("Duplicate rows:", df.duplicated().sum())
        print("Separator: |")
        print("STATUS: PASSED")


Approved Budget
File exists: True
Rows: 80
Columns: ['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']
Missing values: 0
Duplicate rows: 0
Separator: |
STATUS: PASSED

LGEF Utilisation
File exists: True
Rows: 7
Columns: ['financial_year', 'fund_type', 'transaction_type', 'description', 'amount', 'purpose', 'page']
Missing values: 0
Duplicate rows: 0
Separator: |
STATUS: PASSED

Detailed Local Revenue
File exists: True
Rows: 56
Columns: ['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page', 'source_validation_note']
Missing values: 0
Duplicate rows: 0
Separator: |
STATUS: PASSED
